# Horse adapter trimming — cutadapt only

Horse `PRJNA848968` adapter-trim notebook. Removes only sequencing/library adapter contamination plus quality/length filtering. It intentionally does **not** trim EquPD/scFv primer tails; primer/tail trimming is a separate `primer_trim` stage. Writes `results/<dataset>/trimmed/fastq`. No QC is generated here.


In [ ]:
import os, sys, sysconfig, shutil, subprocess
_CONDA_ENV = "/opt/conda/envs/bcr_env"
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
for _site in [_CONDA_ENV + "/lib/python3.11/site-packages", sysconfig.get_path("purelib")]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ["HOME"] = "/data/user/epishkin"
os.environ["XDG_CONFIG_HOME"] = "/data/user/epishkin/.config"
os.makedirs(os.environ["XDG_CONFIG_HOME"], exist_ok=True)


In [ ]:
from pathlib import Path

TRIM_QUALITY = '0,30'
MIN_LENGTH = 250
ADAPTER_TIMES = 2
ADAPTER_MIN_OVERLAP = 10

# PRJNA848968 horse adapter policy:
# Adapter-trim notebook trims ONLY sequencing/library adapter contamination.
# EquPD/scFv primer tails are intentionally NOT cut here; primer/tail trimming
# belongs to primer_trim_horse / MaskPrimers stage.
#
# Source evidence (Rosenfeld 2022 / ENA) does not list exact Illumina adapter
# nucleotide sequences. Empirical fastp detection + cutadapt -O 10 validation on
# raw PRJNA848968 showed run-specific R1 Nextera/i7-primer-like adapters in
# three runs and no reliable R2 adapter. Do not use human TruSeq/NEBNext R1/R2
# adapters for horse by default.
HORSE_R1_ADAPTERS = {
    'SRR19646177': 'CCGAGCCCACGAGACTCCTGAGCATCTCGTATGCCGTCTTCTGCTTG',
    'SRR19646178': 'CCGAGCCCACGAGACCGTACTAGATCTCGTATGCCGTCTTCTGCTTG',
    'SRR19646179': 'CCGAGCCCACGAGACAGGCAGAAATCTCGTATGCCGTCTTCTGCTTG',
    # SRR19646180: no clear R1/R2 adapter detected by fastp/cutadapt smoke tests.
}


In [ ]:
def run_adapter_trim(volume, dataset):
    vol = Path(volume)
    src = vol / 'raw' / dataset
    base = vol / 'results' / dataset / 'trimmed'
    out = base / 'fastq'

    if base.exists():
        shutil.rmtree(base)

    out.mkdir(parents=True, exist_ok=True)

    pairs = sorted(set(f.name.rsplit('_', 1)[0] for f in src.glob('*.fastq.gz')))
    print(f'[adapter_trim_horse] {dataset}: {len(pairs)} pairs')
    print(f'[adapter_trim_horse] quality={TRIM_QUALITY} minlen={MIN_LENGTH} times={ADAPTER_TIMES} overlap={ADAPTER_MIN_OVERLAP}')

    for bn in pairs:
        r1 = src / f'{bn}_1.fastq.gz'
        r2 = src / f'{bn}_2.fastq.gz'
        if not r1.exists() or not r2.exists():
            continue

        o1 = out / f'{bn}_1.trim.fastq.gz'
        o2 = out / f'{bn}_2.trim.fastq.gz'

        ca = [
            'cutadapt',
            '--quality-cutoff', TRIM_QUALITY,
            '--compression-level', '1',
            '-m', str(MIN_LENGTH),
            '--times', str(ADAPTER_TIMES),
            '-O', str(ADAPTER_MIN_OVERLAP),
        ]

        # Adapter-trim stage: do NOT trim EquPD/scFv primer tails here.
        # Those are handled later by primer_trim_horse / MaskPrimers.
        adapter = HORSE_R1_ADAPTERS.get(bn)
        if adapter:
            ca += ['-a', adapter]
        else:
            print(f'  [{bn}] no run-specific R1 adapter; quality/length filtering only')
        ca += ['-o', str(o1), '-p', str(o2), str(r1), str(r2)]

        print(f'  [{bn}] cutadapt ...')
        rr = subprocess.run(ca, capture_output=True, text=True)
        if rr.returncode != 0:
            raise RuntimeError(f'cutadapt failed {bn}: {rr.stderr[:800]}')

    n = len(list(out.glob('*.trim.fastq.gz')))
    print(f'[adapter_trim_horse] DONE: {n} trimmed files')


### Run

Run horse adapter / technical-sequence trimming after reviewing parameters above.

In [ ]:
run_adapter_trim('/data/user/epishkin', 'PRJNA848968')